### Load packages

In [19]:
import pandas as pd
import os
import numpy as np
from datetime import datetime

### Define parameters and file info

In [20]:
# -----------------------------------------------------------------------
# Directory info
# -----------------------------------------------------------------------

github_data_directory = r"..\data\nonprofit_economy"
area990_data_directory = r"..\..\WhenDAFsDrawTheMap\data\pf_grant_matched_csvs"

bmf_directory = r"..\..\..\AppData\Local\Programs\Python\Python310\Lib\site-packages\irsx\CSV\EO BMF Files"
ntee_directory = r"..\..\..\AppData\Local\Programs\Python\Python310\Lib\site-packages\irsx\CSV"

output_directory = r"..\data\nonprofit_economy"

# -----------------------------------------------------------------------
# All-DAF data
# -----------------------------------------------------------------------
pf_file = r"match_output_95_wddm_pf_grants_2023.csv"
pf_path = os.path.join(area990_data_directory, pf_file)
print(
    "PF grants CSV file:",
    pf_path,
    "EXISTS" if os.path.exists(pf_path) else "MISSING"
)

# -----------------------------------------------------------------------
# NTEE data
# -----------------------------------------------------------------------

ntee_file = r"eo_bmf_extract_narrow"
ntee_path = os.path.join(ntee_directory, ntee_file + ".csv")
print(
    "NTEE file:",
    ntee_path,
    "EXISTS" if os.path.exists(ntee_path) else "MISSING"
)

# -----------------------------------------------------------------------
# Output files
# -----------------------------------------------------------------------

pf_summary_file = r"PF_grants_summary_by_recipient_NTEE"
pf_summary_path = os.path.join(output_directory, pf_summary_file + ".csv")
print("PF Summary file:", pf_summary_path)


PF grants CSV file: ..\..\WhenDAFsDrawTheMap\data\pf_grant_matched_csvs\match_output_95_wddm_pf_grants_2023.csv EXISTS
NTEE file: ..\..\..\AppData\Local\Programs\Python\Python310\Lib\site-packages\irsx\CSV\eo_bmf_extract_narrow.csv EXISTS
PF Summary file: ..\data\nonprofit_economy\PF_grants_summary_by_recipient_NTEE.csv


### Load in data

In [21]:
# Load DAF-to-DAF data

pf_raw_df = pd.read_csv(pf_path)

pf_raw_df.head()

,target_id,target_name,matched_candidate_id,matched_candidate_name,matched_attribute1,matched_attribute2,matched_attribute3,append_attribute1,append_attribute2,append_attribute3,similarity_score,matched_scorer
0,10145133,PROUTS NECK ASSOCIATION,NaN,PROUTS NECK ASSOCIATION,2023,NaN,NaN,526358830,WESTWIND FOUNDATION,1000.0,100.0,WRatio
1,10202467,MOUNT DESERT ISLAND BIOLOGICAL LABORATORY,NaN,MOUNT DESERT ISLAND BIOLOGICAL LABORATORY,2023,NaN,NaN,66037106,GERRISH H MILLIKEN FOUNDATION 009209-009,1000.0,100.0,WRatio
2,10211478,HEART MAINE UNITED WAY,NaN,HEART MAINE UNITED WAY,2023,NaN,NaN,43353896,BANGOR SAVINGS BANK FOUNDATION,42450.0,100.0,WRatio
3,10211497,PRESIDENT TRUSTEES COLBY COLLEGE,NaN,PRESIDENT TRUSTEES COLBY COLLEGE,2023,NaN,NaN,46131201,FIDELITY FOUNDATION,500.0,100.0,WRatio
4,10211508,GOOD WILL HOME ASSOCIATION,NaN,GOOD WILL HOME ASSOCIATION,2023,NaN,NaN,10421806,BILL AND JOAN ALFOND FAMILY FOUNDATION,2070.0,100.0,WRatio


In [22]:
# Load in NTEE data

ntee_raw_df = pd.read_csv(ntee_path)

ntee_raw_df.head()

,EIN,LAST_YEAR_IN_BMF,ORG_NAME_RAW,SUBSECTION_CODE,ORG_ADDR_STATE,NTEE_IRS,NTEE_OLD_BMF,NTEE_CODE_RAW,NTEE_CODE_CLEAN,NTEE_COMBINED,NTEE_SOURCE
0,4,2001.0,SCRIPT INC,3.0,OH,NaN,B90,B90,B90,B90,NCCS_CLEAN
1,5,2004.0,JEKYLL ISLAND MUSEUM ASSOCIATES INC,3.0,GA,NaN,A50,A50,A50,A50,NCCS_CLEAN
2,3154,2016.0,OAKLEAF FOREST TENANT MANAGEMENT,3.0,VA,NaN,C36,C36,C36,C36,NCCS_CLEAN
3,4101,2023.0,SOUTH LAFOURCHE QUARTERBACK CLUB,3.0,LA,NaN,N65,N65,N65,N65,NCCS_CLEAN
4,4882,2004.0,LITTLE LEAGUE BASEBALL INC,3.0,MD,NaN,N63,N63,N63,N63,NCCS_CLEAN


### Clean and format columns

In [39]:
# Clean & format columns in pf data

pf_clean_df = pf_raw_df.copy()

# Rename EIN columns
pf_clean_df.rename(
    columns={"target_id": "Recipient EIN", "append_attribute1": "Grantor EIN", "append_attribute3": "Grant Amount"},
   inplace=True
)

# Convert integer columns to integer format
int_cols = ["Grantor EIN", "Recipient EIN"]
pf_clean_df[int_cols] = pf_clean_df[int_cols].astype("Int64")

# Convert grant amount to float
pf_clean_df["Grant Amount"] = pf_clean_df["Grant Amount"].astype(float)

pf_clean_df.head()

,Recipient EIN,target_name,matched_candidate_id,matched_candidate_name,matched_attribute1,matched_attribute2,matched_attribute3,Grantor EIN,append_attribute2,Grant Amount,similarity_score,matched_scorer
0,10145133,PROUTS NECK ASSOCIATION,NaN,PROUTS NECK ASSOCIATION,2023,NaN,NaN,526358830,WESTWIND FOUNDATION,1000.0,100.0,WRatio
1,10202467,MOUNT DESERT ISLAND BIOLOGICAL LABORATORY,NaN,MOUNT DESERT ISLAND BIOLOGICAL LABORATORY,2023,NaN,NaN,66037106,GERRISH H MILLIKEN FOUNDATION 009209-009,1000.0,100.0,WRatio
2,10211478,HEART MAINE UNITED WAY,NaN,HEART MAINE UNITED WAY,2023,NaN,NaN,43353896,BANGOR SAVINGS BANK FOUNDATION,42450.0,100.0,WRatio
3,10211497,PRESIDENT TRUSTEES COLBY COLLEGE,NaN,PRESIDENT TRUSTEES COLBY COLLEGE,2023,NaN,NaN,46131201,FIDELITY FOUNDATION,500.0,100.0,WRatio
4,10211508,GOOD WILL HOME ASSOCIATION,NaN,GOOD WILL HOME ASSOCIATION,2023,NaN,NaN,10421806,BILL AND JOAN ALFOND FAMILY FOUNDATION,2070.0,100.0,WRatio


### Clean and format columns in NTEE data

In [36]:
ntee_raw_df.columns

Index(['EIN', 'LAST_YEAR_IN_BMF', 'ORG_NAME_RAW', 'SUBSECTION_CODE',
       'ORG_ADDR_STATE', 'NTEE_IRS', 'NTEE_OLD_BMF', 'NTEE_CODE_RAW',
       'NTEE_CODE_CLEAN', 'NTEE_COMBINED', 'NTEE_SOURCE'],
      dtype='object')

In [26]:
# Clean & format columns in NTEE data

ntee_clean_df = ntee_raw_df.copy()

# Convert integer columns to integer format
int_cols = ["EIN", "SUBSECTION_CODE"]
ntee_clean_df[int_cols] = ntee_clean_df[int_cols].astype("Int64")

# Drop unnecessary columns
cols_to_drop = ["ORG_ADDR_STATE", "NTEE_OLD_BMF", "NTEE_IRS", "NTEE_CODE_RAW", "NTEE_CODE_CLEAN", "NTEE_SOURCE"]
ntee_clean_df = ntee_clean_df.drop(columns=cols_to_drop)

# Uppercase NTEE_COMBINED
ntee_clean_df["NTEE_COMBINED"] = ntee_clean_df["NTEE_COMBINED"].str.upper()

# Create one-digit NTEE code columm
ntee_clean_df["NTEE_FIRST_DIGIT"] = ntee_clean_df["NTEE_COMBINED"].str[0]

# Identify invalid first digits (not A-Z or null/NaN) and replace with "0"
ntee_clean_df["NTEE_FIRST_DIGIT"] = ntee_clean_df["NTEE_FIRST_DIGIT"].mask(
    ntee_clean_df["NTEE_FIRST_DIGIT"].isna() | ~ntee_clean_df["NTEE_FIRST_DIGIT"].str.match(r"^[A-Z]$", na=False),
    "0"
)

ntee_clean_df.head()

,EIN,LAST_YEAR_IN_BMF,ORG_NAME_RAW,SUBSECTION_CODE,NTEE_COMBINED,NTEE_FIRST_DIGIT
0,4,2001.0,SCRIPT INC,3,B90,B
1,5,2004.0,JEKYLL ISLAND MUSEUM ASSOCIATES INC,3,A50,A
2,3154,2016.0,OAKLEAF FOREST TENANT MANAGEMENT,3,C36,C
3,4101,2023.0,SOUTH LAFOURCHE QUARTERBACK CLUB,3,N65,N
4,4882,2004.0,LITTLE LEAGUE BASEBALL INC,3,N63,N


In [27]:
# Find records where NTEE_COMBINED is null or 0

# Rows where NTEE_FIRST_DIGIT is NaN or not A-Z
invalid_ntee_rows = ntee_clean_df[
    ntee_clean_df["NTEE_FIRST_DIGIT"].isna() |        # null/NaN
    ~ntee_clean_df["NTEE_FIRST_DIGIT"].str.match(r"^[A-Z]$", na=False)  # not a single uppercase letter
]

# print(invalid_ntee_rows.value_counts())
# Show the results
# print(invalid_ntee_rows)
print("Number of invalid NTEE codes:", len(invalid_ntee_rows))

Number of invalid NTEE codes: 64854


In [28]:
# Dedupe the NTEE file

ntee_valid_df = ntee_clean_df.copy()
print("Before deduping:", len(ntee_valid_df))

# Find rows with valid NTEE codes
ntee_valid_df["is_valid_ntee"] = (
    ntee_valid_df["NTEE_FIRST_DIGIT"]
    .astype(str)
    .str.match("^[A-Za-z]$")
)

# Make sure the year is numeric
ntee_valid_df["LAST_YEAR_IN_BMF"] = pd.to_numeric(ntee_valid_df["LAST_YEAR_IN_BMF"], errors="coerce")

# Sort by EIN, valid ntee code flag, and LAST_YEAR_IN_BMF
ntee_sorted_df = ntee_valid_df.sort_values(
    by=["EIN", "is_valid_ntee", "LAST_YEAR_IN_BMF"],
    ascending=[True, False, False]
)

# Keep only the first record for each EIN
ntee_dedupe_df = ntee_sorted_df.drop_duplicates(subset="EIN", keep="first")

# Drop the helper column
ntee_dedupe_df = ntee_dedupe_df.drop(columns=["is_valid_ntee"])

print("After deduping:", len(ntee_dedupe_df))

ntee_dedupe_df.head()

Before deduping: 3687435
After deduping: 3687435


,EIN,LAST_YEAR_IN_BMF,ORG_NAME_RAW,SUBSECTION_CODE,NTEE_COMBINED,NTEE_FIRST_DIGIT
0,4,2001.0,SCRIPT INC,3,B90,B
1,5,2004.0,JEKYLL ISLAND MUSEUM ASSOCIATES INC,3,A50,A
2,3154,2016.0,OAKLEAF FOREST TENANT MANAGEMENT,3,C36,C
3,4101,2023.0,SOUTH LAFOURCHE QUARTERBACK CLUB,3,N65,N
4,4882,2004.0,LITTLE LEAGUE BASEBALL INC,3,N63,N


In [29]:
# Check presence of duplicate rows
duplicates_df = ntee_dedupe_df[ntee_dedupe_df["EIN"].duplicated(keep=False)]

print(f"Total duplicate rows: {len(duplicates_df)}")
# duplicates_df.head(10)

Total duplicate rows: 0


### Add NTEE codes to All-DAF grants file

In [40]:
pf_clean_df.columns

Index(['Recipient EIN', 'target_name', 'matched_candidate_id',
       'matched_candidate_name', 'matched_attribute1', 'matched_attribute2',
       'matched_attribute3', 'Grantor EIN', 'append_attribute2',
       'Grant Amount', 'similarity_score', 'matched_scorer'],
      dtype='object')

In [41]:
# Add NTEE codes to pf file

print("Before merging:", len(pf_clean_df))
print(pf_clean_df["Grant Amount"].sum())

pf_ntee_df = (
    pf_clean_df
    .merge(
        ntee_dedupe_df[["EIN", "NTEE_COMBINED", "NTEE_FIRST_DIGIT"]],
        left_on=["Recipient EIN"],
        right_on=["EIN"],
        how="left",
        indicator=True
    )
)

# Drop helper columns
pf_ntee_df = pf_ntee_df.drop(columns=["_merge"])

# Rename columns
# pf_ntee_df = pf_ntee_df.rename(
#     columns={
#         "NTEE_COMBINED": "GRANTOR_NTEE_COMBINED",
#         "NTEE_FIRST_DIGIT": "GRANTOR_NTEE_FIRST_DIGIT"
#     }
#)

print("After merging:", len(pf_ntee_df))
print(pf_ntee_df["Grant Amount"].sum())

pf_ntee_df.head()


Before merging: 40900
2558979988.0
After merging: 40900
2558979988.0


,Recipient EIN,target_name,matched_candidate_id,matched_candidate_name,matched_attribute1,matched_attribute2,matched_attribute3,Grantor EIN,append_attribute2,Grant Amount,similarity_score,matched_scorer,EIN,NTEE_COMBINED,NTEE_FIRST_DIGIT
0,10145133,PROUTS NECK ASSOCIATION,NaN,PROUTS NECK ASSOCIATION,2023,NaN,NaN,526358830,WESTWIND FOUNDATION,1000.0,100.0,WRatio,10145133,S22,S
1,10202467,MOUNT DESERT ISLAND BIOLOGICAL LABORATORY,NaN,MOUNT DESERT ISLAND BIOLOGICAL LABORATORY,2023,NaN,NaN,66037106,GERRISH H MILLIKEN FOUNDATION 009209-009,1000.0,100.0,WRatio,10202467,U50,U
2,10211478,HEART MAINE UNITED WAY,NaN,HEART MAINE UNITED WAY,2023,NaN,NaN,43353896,BANGOR SAVINGS BANK FOUNDATION,42450.0,100.0,WRatio,10211478,T70,T
3,10211497,PRESIDENT TRUSTEES COLBY COLLEGE,NaN,PRESIDENT TRUSTEES COLBY COLLEGE,2023,NaN,NaN,46131201,FIDELITY FOUNDATION,500.0,100.0,WRatio,10211497,B42,B
4,10211508,GOOD WILL HOME ASSOCIATION,NaN,GOOD WILL HOME ASSOCIATION,2023,NaN,NaN,10421806,BILL AND JOAN ALFOND FAMILY FOUNDATION,2070.0,100.0,WRatio,10211508,P73,P


### Create summaries by NTEE code

In [42]:
# Function to calculate Shonni's summary groupings on 

def shonni_groups(row, first_digit_ntee_code, full_ntee_code):

    code = row[first_digit_ntee_code]
    combined = str(row[full_ntee_code]).upper() if pd.notna(row[full_ntee_code]) else ""
    
    if pd.isna(code):
        return None
    
    code = str(code).upper()

    if combined.startswith(("T30", "T31", "T32")): # private foundations, community foundations, and corporate foundations
        return "Foundations"
    
    if code == "A":
        return "Arts, Culture, Humanities"
    
    elif code == "B":
        if combined.startswith(("B40", "B41", "B43", "B50")):
            return "Colleges and Universities"
        else:
            return "Other Education"
    
    elif code in ["C", "D"]:
        return "Environment and Animals"
    
    elif code in ["E", "F", "G", "H"]:
        if combined.startswith(("E20", "E21", "E22", "E24", "E90", "E91", "E92")):
            return "Hospitals and Nursing Homes"
        else:
            return "Other Health"
    
    elif code in ["I", "J", "K", "L", "M", "N", "O", "P"]:
        return "Human Services"
    
    elif code == "Q":
        return "International, Foreign Affairs"
    
    elif code in ["R", "S", "T", "U", "V", "W"]:
        return "Public, Societal Benefit"
    
    elif code == "X":
        return "Religion Related"
    

    elif code in ["Y", "Z"]:
        return "Unknown, Unclassified"
    
    else:
        return None

In [43]:
pf_ntee_df.columns

Index(['Recipient EIN', 'target_name', 'matched_candidate_id',
       'matched_candidate_name', 'matched_attribute1', 'matched_attribute2',
       'matched_attribute3', 'Grantor EIN', 'append_attribute2',
       'Grant Amount', 'similarity_score', 'matched_scorer', 'EIN',
       'NTEE_COMBINED', 'NTEE_FIRST_DIGIT'],
      dtype='object')

In [44]:
# Apply Shonni groupings to daf-to-daf grants by pf NTEE code

pf_groupings_df = pf_ntee_df.copy()

pf_groupings_df["NTEE_ACTIVITY"] = pf_groupings_df.apply(
    shonni_groups,
    axis=1,
    args=("NTEE_FIRST_DIGIT", "NTEE_COMBINED")
)

pf_groupings_df.head()

,Recipient EIN,target_name,matched_candidate_id,matched_candidate_name,matched_attribute1,matched_attribute2,matched_attribute3,Grantor EIN,append_attribute2,Grant Amount,similarity_score,matched_scorer,EIN,NTEE_COMBINED,NTEE_FIRST_DIGIT,NTEE_ACTIVITY
0,10145133,PROUTS NECK ASSOCIATION,NaN,PROUTS NECK ASSOCIATION,2023,NaN,NaN,526358830,WESTWIND FOUNDATION,1000.0,100.0,WRatio,10145133,S22,S,"Public, Societal Benefit"
1,10202467,MOUNT DESERT ISLAND BIOLOGICAL LABORATORY,NaN,MOUNT DESERT ISLAND BIOLOGICAL LABORATORY,2023,NaN,NaN,66037106,GERRISH H MILLIKEN FOUNDATION 009209-009,1000.0,100.0,WRatio,10202467,U50,U,"Public, Societal Benefit"
2,10211478,HEART MAINE UNITED WAY,NaN,HEART MAINE UNITED WAY,2023,NaN,NaN,43353896,BANGOR SAVINGS BANK FOUNDATION,42450.0,100.0,WRatio,10211478,T70,T,"Public, Societal Benefit"
3,10211497,PRESIDENT TRUSTEES COLBY COLLEGE,NaN,PRESIDENT TRUSTEES COLBY COLLEGE,2023,NaN,NaN,46131201,FIDELITY FOUNDATION,500.0,100.0,WRatio,10211497,B42,B,Other Education
4,10211508,GOOD WILL HOME ASSOCIATION,NaN,GOOD WILL HOME ASSOCIATION,2023,NaN,NaN,10421806,BILL AND JOAN ALFOND FAMILY FOUNDATION,2070.0,100.0,WRatio,10211508,P73,P,Human Services


In [45]:
# Define activity area sort order

activity_order = ["Arts, Culture, Humanities", "Other Education", "Colleges and Universities", "Environment and Animals", "Other Health", 
         "Hospitals and Nursing Homes", "Human Services", "International, Foreign Affairs", "Public, Societal Benefit",
         "Foundations", "Religion Related", "Unknown, Unclassified"]


In [46]:
# Create summary of daf-to-daf data

pf_summary_df = pf_groupings_df.copy()

pf_summary_df["NTEE_ACTIVITY"] = pf_summary_df.apply(
    shonni_groups,
    axis=1,
    args=("NTEE_FIRST_DIGIT", "NTEE_COMBINED")
)

pf_summary = (
    pf_summary_df
    .groupby("NTEE_ACTIVITY", as_index=False)
    .agg(
        Count_of_Orgs=("EIN", "count"),
        Grant_Amount=("Grant Amount", "sum")
    )
)

pf_summary["NTEE_ACTIVITY"] = pd.Categorical(
    pf_summary["NTEE_ACTIVITY"],
    categories=activity_order,
    ordered=True
)

pf_summary = pf_summary.sort_values("NTEE_ACTIVITY")

pf_summary

,NTEE_ACTIVITY,Count_of_Orgs,Grant_Amount
0,"Arts, Culture, Humanities",2570,119781936.0
7,Other Education,6936,310207004.0
1,Colleges and Universities,506,42995908.0
2,Environment and Animals,2078,216598395.0
8,Other Health,4814,313793815.0
4,Hospitals and Nursing Homes,1385,91927309.0
5,Human Services,9809,327456013.0
6,"International, Foreign Affairs",1031,92386810.0
9,"Public, Societal Benefit",6834,614078460.0
3,Foundations,2535,323798750.0


### Output results

In [47]:
# Output the pf summary table to csv

# Archive old output file if present

if os.path.exists(pf_summary_path):

    # Get modification time of existing file
    old_timestamp = datetime.fromtimestamp(
        os.path.getmtime(pf_summary_path)
    ).strftime("%Y%m%d_%H%M%S")

    archived_filename = (
        f"{pf_summary_file}_archived_{old_timestamp}.csv"
    )

    archived_path = os.path.join(
        output_directory,
        archived_filename
    )

    os.rename(
        pf_summary_path,
        archived_path
    )

    print(
        "Archived existing file:",
        archived_path
    )

pf_summary.to_csv(pf_summary_path, index=False)

Archived existing file: ..\data\nonprofit_economy\PF_grants_summary_by_recipient_NTEE_archived_20260701_160148.csv
